# Few-shot RETFound → DeepDRiD

Adapt checkpoint grading sang target domain DeepDRiD Regular Fundus với đúng K ảnh có nhãn mỗi grade. Chạy **Cell 1–7**, sau đó chọn đúng một trong: Train mới, Resume hoặc Test cuối.

In [ ]:
# CELL 1 — GPU
import torch, sys, platform
from datetime import datetime, timezone
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime → Change runtime type → T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

In [ ]:
# CELL 2 — Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/dr_fewshot')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
# CELL 3 — Source project
import os, subprocess
GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'fix/semi-pseudo-weight-batch4'
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/train_fewshot/requirements.txt'], check=True)
print('Project ready:', REPO_DIR)

## Dataset target đã chọn

Notebook dùng **DeepDRiD v1.1 Regular Fundus** chính thức từ `deepdrdoc/DeepDRiD`, không dùng Kaggle mirror. Dữ liệu tải một lần vào Drive và được tái sử dụng.

In [ ]:
# CELL 4 — Download/cache DeepDRiD official v1.1
import shutil
from ai.train_fewshot.runtime import prepare_deepdrid_target
DEEPDRID_RAW = DRIVE_ROOT / 'datasets' / 'DeepDRiD-v1.1'
DEEPDRID_PREPARED = DRIVE_ROOT / 'datasets' / 'DeepDRiD-v1.1-prepared'
if not DEEPDRID_RAW.exists():
    DEEPDRID_RAW.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'v1.1',
        'https://github.com/deepdrdoc/DeepDRiD.git', str(DEEPDRID_RAW)
    ], check=True)
    subprocess.run(['git', '-C', str(DEEPDRID_RAW), 'lfs', 'pull'], check=False)
else:
    print('Dùng lại DeepDRiD đã tải:', DEEPDRID_RAW)
TARGET_DATASET_DIR = prepare_deepdrid_target(DEEPDRID_RAW, DEEPDRID_PREPARED)
print('Prepared target dataset:', TARGET_DATASET_DIR)

In [ ]:
# CELL 5 — Cấu hình
GRADE_CHECKPOINT = Path('/content/drive/MyDrive/retfound_merged_seed42/checkpoint-best.pth')
RUN_NAME = 'fewshot_deepdrid_5shot_seed42'
OUTPUT_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
FEWSHOT_CONFIG = {
    'shots': 5, 'queries': 1, 'epochs': 8, 'patience': 3,
    'train_episodes': 20, 'embedding_dim': 0,
    'encoder_lr': 1e-6, 'projection_lr': 1e-4,
    'weight_decay': 0.05, 'unfreeze_last_blocks': 1,
    'forward_batch_size': 2, 'temperature': 0.1, 'seed': 42,
}
if not GRADE_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Không tìm thấy checkpoint grade: {GRADE_CHECKPOINT}')
print('Checkpoint:', GRADE_CHECKPOINT)
print('Target:', TARGET_DATASET_DIR)
print('Output:', OUTPUT_DIR)
print('Config:', FEWSHOT_CONFIG)

In [ ]:
# CELL 6 — Kiểm tra checkpoint grading contract
import json
state = torch.load(GRADE_CHECKPOINT, map_location='cpu', weights_only=False)
grade_args = state.get('args', {})
required = ['model_source', 'image_size', 'preprocessing', 'loss']
missing = [key for key in required if key not in grade_args]
if missing:
    raise ValueError(f'Checkpoint thiếu metadata: {missing}')
print(json.dumps({key: grade_args.get(key) for key in required + ['architecture', 'model_name']}, indent=2))
del state

In [ ]:
# CELL 7 — Command và live log
import shlex, signal
def build_command(*, resume=None, eval_only=False):
    c = FEWSHOT_CONFIG
    command = [
        sys.executable, '-u', '-m', 'ai.train_fewshot.train',
        '--checkpoint', str(GRADE_CHECKPOINT),
        '--target-dataset-dir', str(TARGET_DATASET_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--shots', str(c['shots']), '--queries', str(c['queries']),
        '--epochs', str(c['epochs']), '--patience', str(c['patience']),
        '--train-episodes', str(c['train_episodes']),
        '--embedding-dim', str(c['embedding_dim']),
        '--encoder-lr', str(c['encoder_lr']), '--projection-lr', str(c['projection_lr']),
        '--weight-decay', str(c['weight_decay']),
        '--unfreeze-last-blocks', str(c['unfreeze_last_blocks']),
        '--forward-batch-size', str(c['forward_batch_size']),
        '--temperature', str(c['temperature']), '--seed', str(c['seed']),
    ]
    if resume is not None: command.extend(['--resume', str(resume)])
    if eval_only: command.append('--eval-only')
    return command

def run_streaming(command, log_name):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    log_path = OUTPUT_DIR / log_name
    print('COMMAND:', shlex.join(command))
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with log_path.open('a', encoding='utf-8') as handle:
            for line in process.stdout:
                timestamped = f'[{datetime.now(timezone.utc).isoformat()}] {line}'
                print(timestamped, end='', flush=True)
                handle.write(timestamped); handle.flush()
        process.wait()
    except KeyboardInterrupt:
        print('\nĐang dừng; checkpoint epoch hoàn tất gần nhất vẫn được giữ.')
        process.send_signal(signal.SIGINT)
        process.wait(timeout=30)
    if process.returncode not in (0, 130, -2):
        raise RuntimeError(f'Process exit code {process.returncode}')

## Train mới

In [ ]:
# TRAIN NEW
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError('Run đã tồn tại; hãy Resume hoặc đổi RUN_NAME.')
run_streaming(build_command(), 'notebook-train-live.log')

## Resume sau khi Colab ngắt

In [ ]:
# RESUME FEWSHOT
LAST_CHECKPOINT = OUTPUT_DIR / 'checkpoint-last.pth'
if not LAST_CHECKPOINT.is_file(): raise FileNotFoundError(LAST_CHECKPOINT)
run_streaming(build_command(resume=LAST_CHECKPOINT), 'notebook-resume-live.log')

## Test cuối

Chỉ cell này mới đọc target test. Metric test không dùng để train, early stopping hoặc chọn checkpoint.

In [ ]:
# TEST HELD-OUT TARGET
TEST_KIND = 'best'  # best | last
TEST_CHECKPOINT = OUTPUT_DIR / f'checkpoint-{TEST_KIND}.pth'
if not TEST_CHECKPOINT.is_file(): raise FileNotFoundError(TEST_CHECKPOINT)
run_streaming(build_command(resume=TEST_CHECKPOINT, eval_only=True), 'notebook-test-live.log')
comparison = OUTPUT_DIR / 'comparison.json'
if comparison.is_file(): print(comparison.read_text(encoding='utf-8'))